# S3DF Pipeline Development Notebook (Likelihood-Based Loss)

This notebook replicates the 5-stage optimization pipeline using **likelihood-based loss**:
- Poisson NLL for charges
- First-arrival NLL for times

**Key difference from tracking_opt_development.ipynb**: Time treatment follows `grad_loss_and_opt_in_2D_likelihood.ipynb`
where observed times are shifted by t0 and the track is simulated with t0=0.

**5-Stage Pipeline:**
1. Stage 0: Energy estimation via scan at origin
2. Stage 1: Hierarchical grid search for position + t0
3. Stage 2: Hierarchical cone direction search
4. Stage 3: Energy scan optimization
5. Stage 4: Adam optimizer refinement

## Cell 1: Environment Setup and Imports

In [ ]:
import sys
sys.path.append('..')

# Standard imports
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from tqdm import tqdm
import json
import os
import time
import pickle
import glob
import uproot
from pathlib import Path
from jax import jit, value_and_grad
from jax.scipy.special import gammaln
import optax
import subprocess

# LUCiD imports
from lucid.geometry import generate_detector
from lucid.generate import read_photon_data_from_photonsim
from lucid.simulation import setup_event_simulator
from lucid.utils import load_range_params, check_track_endpoint_in_detector
from lucid.detector_params import ParticleParams, load_detector_params

# Optimization imports
from lucid.optimization.grid_search import (
    load_optimization_config, 
    get_detector_bounds, 
    hierarchical_position_grid_search
)
from lucid.optimization.utils.functions import (
    hierarchical_direction_search_cone, 
    energy_scan_optimization,
    cartesian_to_spherical, 
    spherical_to_cartesian, 
    performance_summary,
    estimate_muon_energy_from_photon_count
)
from lucid.losses import (
    first_arrival_nll, 
    origin_time_loss,
    get_optimal_tau_vtx,
    TAU_VTX_PARAM_A,
    TAU_VTX_PARAM_B,
    TAU_VTX_PARAM_C,
)
from lucid.optimization.run import load_config

PHYSICS_CONFIG = '../config/SK_physics_config.json'

print("Environment setup complete")
print(f"Tau vtx parametrization: tau = {TAU_VTX_PARAM_A:.6e}*Nrays + {TAU_VTX_PARAM_B:.6e}*E + {TAU_VTX_PARAM_C:.4f}")

## Cell 2: Configuration Selection

In [ ]:
# =====================================================================
# CONFIG PARAMETERIZATION - Change this to switch configs (0-8)
# =====================================================================
CONFIG_INDEX = 5

config_dir = Path('../s3df_jobs/nrays_config')
config_path = config_dir / f'opt_config_{CONFIG_INDEX}.json'
script_path = config_dir / 'create_configs.py'

if not config_path.exists():
    print("Config file not found. Creating it...")
    subprocess.run(
        [sys.executable, script_path.name],
        cwd=config_dir,
        check=True
    )
    print("Config file successfully created.")
else:
    print("Config file already exists.")

print(config_path)
adam_config = load_config(config_path)

# Load configuration
config = load_optimization_config(config_path)

# Add default values for parameters if not present
if 'optimization_params' not in config:
    config['optimization_params'] = {}
config['optimization_params'].setdefault('damping_factor', 0.998)

if 'adam_optimizer' not in config:
    config['adam_optimizer'] = {}
config['adam_optimizer'].setdefault('learning_rate', 0.2)
config['adam_optimizer'].setdefault('b1', 0.9)
config['adam_optimizer'].setdefault('b2', 0.999)
config['adam_optimizer'].setdefault('eps', 1e-8)

# Display key parameters
print(f"Loaded Config {CONFIG_INDEX}")
print("=" * 50)
print(f"nphot:        {config['basic_config']['nphot']:,}")
print(f"n_events:     {config['basic_config']['n_events']}")
print(f"temperature:  {config['basic_config']['temperature']}")
print(f"k:            {config['basic_config']['k']}")
print(f"c_medium:     {config['basic_config']['c_medium']:.6f}")

## Cell 3: Detector and Simulator Setup

In [ ]:
# Extract basic configuration
default_json_filename = '../config/SK_geom_config.json'
data_dir = '/sdf/data/neutrino/cjesus/photonsim_output/water/monoenergetic/event_by_event/mu-/1500MeV/'#'../data/water/muon/'
#data_dir = '../data/water/muon/'
TEMPERATURE = 0.05
TEMPERATURE = 0.1
N_EVENTS = 20
K = config['basic_config']['k']
Nphot = 50_000
C_MEDIUM = config['basic_config']['c_medium']
TAU = 0.15  # Time constant for first-arrival NLL

# Setup detector
print(f"Setting up detector from: {default_json_filename}")
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)

# Get detector bounds
detector_bounds = get_detector_bounds(detector)
DETECTOR_R = detector_bounds.get('r', None)
DETECTOR_H = detector_bounds.get('H', None)

print(f"Detector type: {detector_bounds['type']}")
print(f"Detector R: {DETECTOR_R:.2f} m")
print(f"Detector H: {DETECTOR_H:.2f} m")
print(f"Number of sensors: {NUM_DETECTORS}")

# Load range parametrization for track endpoint validation
range_params = load_range_params('muon', 'water')
print(f"Loaded range parametrization: {range_params['description']}")

# Setup simulators
print("\nSetting up simulators...")
prediction_simulator = setup_event_simulator(
    default_json_filename, Nphot, TEMPERATURE, 
    max_sensors_per_cell=4, K=K, is_data=False, hit_mode='per_photon',
    physics_config=PHYSICS_CONFIG, default_detector_params=True
)

data_simulator = setup_event_simulator(
    default_json_filename, Nphot, temperature=0.0, 
    K=K, is_data=True, is_calibration=False,
    physics_config=PHYSICS_CONFIG, default_detector_params=True
)

# Load detector parameters
detector_params = load_detector_params(PHYSICS_CONFIG)

print("Simulators ready!")
print(f"\nDetector params: scatter_length={detector_params.scatter_length}, "
      f"wall_reflection_rate={detector_params.wall_reflection_rate}, "
      f"absorption_length={detector_params.absorption_length}, qe={detector_params.qe}")

## Cell 4: Event Generation Function

In [ ]:
def generate_event_data(event_idx, random_key, data_dir, data_simulator,
                       detector_bounds, fraction=0.9):
    """
    Generate a single event with random parameters within detector bounds.
    """
    # Get all .root files in the directory
    root_files = sorted(glob.glob(os.path.join(data_dir, "*.root")))
    if not root_files:
        raise ValueError(f"No .root files found in directory: {data_dir}")

    # Randomly select a file
    file_select_key, random_key = jax.random.split(random_key)
    file_idx = jax.random.randint(file_select_key, shape=(), minval=0, maxval=len(root_files))
    data_file = root_files[int(file_idx)]

    # Get number of entries
    with uproot.open(data_file) as file:
        tree = file['OpticalPhotons']
        n_entries = tree.num_entries

    entry_idx = event_idx % n_entries
    photon_data = read_photon_data_from_photonsim(data_file, entry_idx)

    # Process photon data
    photon_origins = photon_data['photon_origins']
    photon_directions = photon_data['photon_directions']
    photon_times = photon_data['photon_times']
    N = len(photon_origins)
    
    # Padding to 1_000_000
    padding_size = max(0, 1_000_000 - N)
    photon_data['photon_origins'] = jnp.pad(photon_origins, ((0, padding_size), (0, 0)), 
                                        mode='constant', constant_values=0)

    default_direction = jnp.array([0.0, 0.0, 1.0])
    padding_directions = jnp.tile(default_direction, (padding_size, 1))
    if padding_size > 0:
        photon_data['photon_directions'] = jnp.concatenate([photon_directions, padding_directions], axis=0)
    else:
        photon_data['photon_directions'] = photon_directions

    photon_data['photon_times'] = jnp.pad(photon_times, (0, padding_size),
                                          mode='constant', constant_values=0)
    photon_data['N'] = N

    key = random_key
    DETECTOR_R = detector_bounds['r']
    DETECTOR_H = detector_bounds['H']

    # Random position
    r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=DETECTOR_R * fraction)
    key, _ = jax.random.split(key)
    theta = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    z_vert = jax.random.uniform(key, shape=(), minval=-DETECTOR_H/2 * fraction,
                               maxval=DETECTOR_H/2 * fraction)
    true_position = jnp.array([r_vert * jnp.cos(theta), r_vert * jnp.sin(theta), z_vert])

    # Random direction
    key, _ = jax.random.split(key)
    phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
    sin_theta = jnp.sqrt(1 - cos_theta**2)
    true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

    true_energy = photon_data['energy']
    TRUE_T0 = jax.random.uniform(key, shape=(), minval=-15.0, maxval=15.0)

    true_track = ParticleParams.from_cartesian(
        energy=true_energy, position=true_position, direction=true_direction, t0=0.0
    )

    # Compute rotation
    original_direction = jnp.array([0.0, 0.0, 1.0])
    true_direction_norm = true_direction / (jnp.linalg.norm(true_direction) + 1e-8)
    rotation_axis = jnp.cross(original_direction, true_direction_norm)
    axis_norm = jnp.linalg.norm(rotation_axis)
    rotation_axis = jnp.where(
        axis_norm < 1e-6,
        jnp.array([1.0, 0.0, 0.0]),
        rotation_axis / (axis_norm + 1e-8)
    )
    rotation_angle = jnp.arccos(jnp.clip(
        jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
    ))
    
    photon_data['rotation_axis'] = rotation_axis
    photon_data['rotation_angle'] = rotation_angle
    photon_data['apply_rotation'] = jnp.array(True)
    photon_data['apply_translation'] = jnp.array(True)
    photon_data['translation_vector'] = true_position
    
    key, _ = jax.random.split(key)
    true_data = jax.lax.stop_gradient(data_simulator(true_track, key, photon_data))
    
    hit_counts, hit_times_raw = true_data
    hit_times = hit_times_raw + TRUE_T0

    return {
        'event_idx': event_idx,
        'entry_idx': entry_idx,
        'true_energy': float(true_energy),
        'true_position': np.array(true_position),
        'true_direction': np.array(true_direction),
        'TRUE_T0': float(TRUE_T0),
        'true_data': true_data,
        'hit_times': hit_times,
        'hit_counts': hit_counts,
        'photon_data': photon_data
    }

print("Event generation function defined.")

## Cell 5: Likelihood-Based Loss Function Definition

This is the key difference from `tracking_opt_development.ipynb`:
- Uses `poisson_nll` for charge comparison
- Uses `first_arrival_nll` for time comparison
- **Time treatment**: Observed times are shifted by t0, track simulated with t0=0

In [ ]:
# =====================================================================
# Likelihood-based loss functions (matching run_eval_with_parametrization.py)
# =====================================================================

def poisson_nll(true, pred, eps=1e-8):
    """Poisson negative log-likelihood, normalized by total true counts."""
    nll = pred - true * jnp.log(pred + eps) + gammaln(true + 1.0)
    return jnp.sum(nll) / (jnp.sum(true) + eps)

def energy_loss(simulated_counts, true_counts, eps=1e-8):
    """
    Energy loss using log ratio of total counts.
    Use this for initial energy guesses - only cares about total light yield,
    not spatial distribution.
    """
    total_true = jnp.sum(true_counts)
    total_sim = jnp.sum(simulated_counts)
    return jnp.abs(jnp.log(total_sim / (total_true + eps)))

def softplus(x):
    return jnp.log1p(jnp.exp(-jnp.abs(x))) + jnp.maximum(x, 0.)

def smooth_pinball(r, tau=0.1, sigma=0.5):
    """
    Smooth version of pinball/quantile loss.
    Minimizer sets approx quantile_tau(r) ~= 0.
    """
    pos = softplus(r / sigma) * sigma
    neg = softplus(-r / sigma) * sigma
    return tau * pos + (1.0 - tau) * neg

@jit
def origin_time_loss_configurable(origin, detector_positions, true_times, true_q, t0,
                                   tau_vtx, photosensor_radius=0.25, c_medium=(0.299792/1.33)):
    """Origin time loss with configurable tau_vtx parameter."""
    d = jnp.linalg.norm(detector_positions - origin[None, :], axis=1)
    expected = (d - photosensor_radius) / c_medium
    r = true_times - expected - t0
    w = jnp.where(true_q > 0., 1., 0.)
    wsum = jnp.sum(w) + 1e-8
    main = jnp.sum(w * smooth_pinball(r, tau=tau_vtx, sigma=0.25)) / wsum
    return main


def create_combined_loss_function(prediction_simulator, num_detectors, tau_time, 
                                  nrays, detector_positions):
    """
    Create likelihood-based combined loss function with DYNAMIC tau_vtx.
    
    This matches the loss used in run_eval_with_parametrization.py:
    - Poisson NLL for charges
    - First-arrival NLL for times
    - Origin time loss (vertex) with energy-dependent tau_vtx
    
    tau_vtx is computed from the current reconstructed energy using the
    learned parametrization, with stop_gradient applied.
    """
    nrays_float = float(nrays)
    
    @jit
    def combined_product_loss(params, observed_times, observed_counts, key):
        """
        Likelihood-based loss with 3-term combined loss.
        
        Args:
            params: [x, y, z, t0, theta, phi, energy]
        
        Returns:
            combined_loss, (charge_loss, time_loss, vertex_loss, sqrt_ct, tau_vtx)
        """
        position = params[:3]
        t0 = params[3]
        theta = params[4]
        phi = params[5]
        energy = params[6]

        # Create track with t0=0 (time shift is applied to observations)
        track = ParticleParams(energy=energy, position=position, theta=theta, phi=phi, t0=jnp.array(0.0))
        log_w, flat_times, flat_indices, total_charge = prediction_simulator(track, key)

        # Poisson NLL for charges
        charge_loss = poisson_nll(observed_counts, total_charge)

        # Shift observed times by t0
        t_obs_shifted = observed_times - t0
        
        # First-arrival NLL for times
        time_nll = first_arrival_nll(
            log_w, flat_times, flat_indices,
            t_obs_shifted, tau_time, num_detectors)
        
        # Mask for hit sensors
        hit_mask = observed_counts > 0
        n_hit = jnp.sum(hit_mask) + 1e-8
        time_loss = jnp.sum(jnp.where(hit_mask, time_nll, 0.0)) / n_hit

        # DYNAMIC tau_vtx: compute from current energy with stop_gradient
        tau_vtx = jax.lax.stop_gradient(
            TAU_VTX_PARAM_A * nrays_float + TAU_VTX_PARAM_B * energy + TAU_VTX_PARAM_C
        )
        tau_vtx = jnp.clip(tau_vtx, 0.05, 0.95)

        # Vertex loss with configurable tau_vtx
        vertex_loss_val = origin_time_loss_configurable(
            jax.lax.stop_gradient(position), detector_positions, observed_times,
            observed_counts, t0, tau_vtx=tau_vtx)

        c = charge_loss
        t = time_loss
        v = vertex_loss_val 
        s = 0.

        # 3-term combined loss (matching tau scan scripts)
        combined = jnp.sqrt((c + s) * (t + s) * (v + s))\
            + jnp.sqrt((c + s) * jax.lax.stop_gradient((t + s) * (v + s)))\
            + jnp.sqrt((v + s) * jax.lax.stop_gradient((t + s) * (c + s)))

        sqrt_ct = jnp.sqrt(c * t)
        
        return combined, (charge_loss, time_loss, vertex_loss_val, sqrt_ct, tau_vtx)

    
    @jit
    def combined_product_loss_landscape(params, observed_times, observed_counts, key):
        """Wrapper returning only the combined loss (for optimization landscapes)."""
        combined_loss, _ = combined_product_loss(params, observed_times, observed_counts, key)
        return combined_loss

    combined_grad_fn = jit(value_and_grad(combined_product_loss, has_aux=True))
    return combined_grad_fn, combined_product_loss, combined_product_loss_landscape


# Create combined gradient function
combined_grad_fn, combined_product_loss, combined_loss_landscape = create_combined_loss_function(
    prediction_simulator, NUM_DETECTORS, TAU, Nphot, detector_points
)

print("Likelihood-based loss function created.")
print(f"  - Using TAU_TIME={TAU} for first-arrival NLL")
print(f"  - Using dynamic tau_vtx from parametrization for vertex loss")
print(f"  - 3-term combined loss: sqrt(c*t*v) + sqrt(c*sg(t*v)) + sqrt(v*sg(t*c))")
print(f"  - energy_loss for initial guesses (log ratio of total counts)")

## Cell 6: Stage 0 - Energy Estimation at Origin

In [ ]:
def run_stage_0_energy_estimation_likelihood(observed_times, observed_counts,
                                             true_energy, verbosity=2):
    """
    Stage 0: Energy estimation via scan at origin position.
    
    Uses energy_loss (log ratio of total counts) - only cares about matching
    total light yield, not spatial distribution.
    """
    if verbosity >= 2:
        print("=" * 60)
        print("STAGE 0: Energy Estimation at Origin (energy_loss)")
        print("=" * 60)
    
    # Scan energy at origin with standard direction
    theta_init = jnp.arccos(1/jnp.sqrt(3))
    phi_init = jnp.pi/4.
    position_init = jnp.array([0., 0., 0.])
    t0_init = 0.
    
    energy_guess = 1000 + np.random.uniform(-50, 50)
    energy_delta = 700
    n_steps = 10
    
    energies = jnp.linspace(energy_guess - energy_delta, energy_guess + energy_delta, n_steps)
    
    scan_results = []
    best_loss = float('inf')
    best_energy = energy_guess
    
    scan_key = jax.random.PRNGKey(42)
    
    for energy in energies:
        # Use energy_loss (log ratio) for initial energy guess
        track = ParticleParams(energy=energy, position=position_init, 
                              theta=theta_init, phi=phi_init, t0=jnp.array(0.0))
        _, _, _, total_charge = prediction_simulator(track, scan_key)
        loss = energy_loss(total_charge, observed_counts)
        
        scan_results.append({'energy': float(energy), 'loss': float(loss)})
        
        if loss < best_loss:
            best_loss = loss
            best_energy = energy
    
    if verbosity >= 2:
        print(f"\n  Energy guess: {best_energy:.1f} MeV (true: {true_energy:.1f} MeV)")
        print(f"  Energy error: {abs(best_energy - true_energy):.1f} MeV")
    
    return {
        'best_energy': float(best_energy),
        'best_loss': float(best_loss),
        'scan_results': scan_results
    }


def visualize_stage_0(stage0_results, true_energy):
    """Visualization: Energy vs Loss curve"""
    energies = [r['energy'] for r in stage0_results['scan_results']]
    losses = [r['loss'] for r in stage0_results['scan_results']]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(energies, losses, 'b-o', label='Energy Scan Loss', markersize=8)
    ax.axvline(true_energy, color='r', linestyle='--', linewidth=2, label=f'True E={true_energy:.0f} MeV')
    ax.axvline(stage0_results['best_energy'], color='g', linestyle='--', linewidth=2, 
               label=f'Best E={stage0_results["best_energy"]:.0f} MeV')
    ax.set_xlabel('Energy (MeV)', fontsize=12)
    ax.set_ylabel('energy_loss (log ratio)', fontsize=12)
    ax.legend(fontsize=11)
    ax.set_title('Stage 0: Initial Energy Scan at Origin', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return fig

print("Stage 0 functions defined (using energy_loss for initial guess).")

## Cell 7: Stage 1 - Hierarchical Position + t0 Grid Search

In [ ]:
def run_stage_1_position_search(hit_detector_positions, observed_times, observed_counts,
                                 true_position, TRUE_T0, initial_t0, verbosity=2):
    """
    Stage 1: Hierarchical grid search for position + t0
    (Uses origin_time_loss which doesn't depend on the full likelihood)
    """
    if verbosity >= 2:
        print("\n" + "=" * 60)
        print("STAGE 1: Hierarchical Position + t0 Grid Search")
        print("=" * 60)
    
    pos_results = hierarchical_position_grid_search(
        hit_detector_positions, observed_times, observed_counts,
        true_position, TRUE_T0, initial_t0, detector_bounds,
        n_div=config['position_grid_search']['pos_n_div'],
        t0_n_div=config['position_grid_search']['t0_n_div'],
        levels=config['position_grid_search']['pos_levels'],
        fraction=config['position_grid_search']['pos_fraction'],
        t0_min=config['position_grid_search']['t0_min'],
        t0_max=config['position_grid_search']['t0_max'],
        min_L=config['position_grid_search']['pos_min_L'],
        verbosity=verbosity
    )
    
    if verbosity >= 2:
        print(f"\n  Best position: {pos_results['best_position']}")
        print(f"  Best t0: {pos_results['best_t0']:.3f} (true: {TRUE_T0:.3f})")
        print(f"  Position error: {pos_results['position_error']:.3f} m")
        print(f"  t0 error: {pos_results['t0_error']:.3f}")
    
    return pos_results

print("Stage 1 functions defined.")

## Cell 8: Stage 2 - Hierarchical Cone Direction Search

In [ ]:
# from lucid.optimization.utils.functions import hierarchical_direction_search_cone

# def run_stage_2_direction_search_likelihood(optimal_position, optimal_t0, energy_guess,
#                                   hit_detector_positions, observed_times,
#                                   observed_counts, true_data, true_direction, verbosity=2):
#     """
#     Stage 2: Hierarchical cone-based direction search
#     """
#     if verbosity >= 2:
#         print("\n" + "=" * 60)
#         print("STAGE 2: Hierarchical Cone Direction Search")
#         print("=" * 60)
    
#     cone_results = hierarchical_direction_search_cone(
#         prediction_simulator, optimal_position, optimal_t0,
#         hit_detector_positions, observed_times, observed_counts,
#         true_data, energy_guess,
#         levels=config['cone_direction_search']['cone_levels'],
#         initial_div=config['cone_direction_search']['cone_initial_div'],
#         max_angle_deg=config['cone_direction_search']['cone_max_angle_deg'],
#         reduction=config['cone_direction_search']['cone_reduction'],
#         verbosity=verbosity
#     )
    
#     # Calculate direction error
#     best_dir = cone_results['best_direction']
#     cos_angle = np.clip(np.dot(best_dir, true_direction), -1.0, 1.0)
#     direction_error = np.degrees(np.arccos(cos_angle))
#     cone_results['direction_error'] = direction_error
    
#     if verbosity >= 2:
#         print(f"\n  Best direction: {best_dir}")
#         print(f"  True direction: {true_direction}")
#         print(f"  Direction error: {direction_error:.2f} degrees")
    
#     return cone_results

In [ ]:
# def run_stage_2_direction_search_likelihood(optimal_position, optimal_t0, energy_guess,
#                                             observed_times, observed_counts, 
#                                             true_direction, verbosity=2):
#     """
#     Stage 2: Hierarchical cone-based direction search using likelihood loss.
#     """
#     if verbosity >= 2:
#         print("\n" + "=" * 60)
#         print("STAGE 2: Hierarchical Cone Direction Search (Likelihood)")
#         print("=" * 60)
    
#     levels = config['cone_direction_search']['cone_levels']
#     initial_div = config['cone_direction_search']['cone_initial_div']
#     max_angle_deg = config['cone_direction_search']['cone_max_angle_deg']
#     reduction = config['cone_direction_search']['cone_reduction']
    
#     # Initial direction (uniform on sphere center)
#     best_direction = np.array([0., 0., 1.])
#     best_theta = 0.
#     best_phi = 0.
#     best_loss = float('inf')
    
#     search_path = []
#     cone_key = jax.random.PRNGKey(42)
    
#     current_max_angle = np.radians(max_angle_deg)
    
#     for level in range(levels):
#         # Generate directions in cone around current best
#         n_theta = initial_div
#         n_phi = initial_div * 2
        
#         directions = []
#         direction_results = []
        
#         # Generate cone samples
#         for i in range(n_theta):
#             theta_offset = current_max_angle * (i / max(n_theta - 1, 1))
#             for j in range(n_phi):
#                 phi_offset = 2 * np.pi * (j / n_phi)
                
#                 # Convert to direction
#                 if level == 0:
#                     theta = theta_offset
#                     phi = phi_offset
#                 else:
#                     # Rotate around best direction
#                     theta = best_theta + theta_offset * np.cos(phi_offset)
#                     phi = best_phi + theta_offset * np.sin(phi_offset)
                
#                 theta = np.clip(theta, 0, np.pi)
#                 phi = np.mod(phi, 2 * np.pi)
                
#                 direction = spherical_to_cartesian(theta, phi)
#                 directions.append(direction)
                
#                 # Evaluate loss
#                 params = jnp.array([optimal_position[0], optimal_position[1], optimal_position[2],
#                                    optimal_t0, theta, phi, energy_guess])

#                 track = ParticleParams(energy=energy_guess, position=optimal_position, theta=theta, phi=phi, t0=optimal_t0)
#                 _, _, _, total_charge = prediction_simulator(track, cone_key)
#                 loss = poisson_nll(observed_counts, total_charge)#combined_product_loss(params, observed_times, observed_counts, cone_key)
                
#                 direction_results.append({
#                     'direction': np.array(direction),
#                     'theta': float(theta),
#                     'phi': float(phi),
#                     'loss': float(loss)
#                 })
                
#                 if loss < best_loss:
#                     best_loss = loss
#                     best_direction = np.array(direction)
#                     best_theta = theta
#                     best_phi = phi
        
#         search_path.append({
#             'level': level,
#             'directions': np.array(directions),
#             'direction_results': direction_results,
#             'best_direction': best_direction.copy(),
#             'best_loss': best_loss
#         })
        
#         # Reduce cone angle for next level
#         current_max_angle *= reduction
        
#         if verbosity >= 2:
#             print(f"  Level {level}: best_loss={best_loss:.6f}")
    
#     # Calculate direction error
#     cos_angle = np.clip(np.dot(best_direction, true_direction), -1.0, 1.0)
#     direction_error = np.degrees(np.arccos(cos_angle))
    
#     if verbosity >= 2:
#         print(f"\n  Best direction: {best_direction}")
#         print(f"  True direction: {true_direction}")
#         print(f"  Direction error: {direction_error:.2f} degrees")
    
#     return {
#         'best_direction': best_direction,
#         'best_theta': float(best_theta),
#         'best_phi': float(best_phi),
#         'best_loss': float(best_loss),
#         'direction_error': direction_error,
#         'search_path': search_path
#     }

# print("Stage 2 functions defined.")

In [ ]:
def run_stage_2_direction_search_likelihood(optimal_position, optimal_t0, energy_guess,
                                            observed_times, observed_counts, 
                                            true_direction, verbosity=2):
    """
    Stage 2: Hierarchical cone-based direction search using likelihood loss.
    """
    if verbosity >= 2:
        print("\n" + "=" * 60)
        print("STAGE 2: Hierarchical Cone Direction Search (Likelihood)")
        print("=" * 60)

    levels = config['cone_direction_search']['cone_levels']
    initial_div = config['cone_direction_search']['cone_initial_div']
    max_angle_deg = config['cone_direction_search']['cone_max_angle_deg']
    reduction = config['cone_direction_search']['cone_reduction']

    # Initial direction
    best_direction = np.array([0., 0., 1.])
    best_theta = 0.
    best_phi = 0.
    best_loss = float('inf')

    search_path = []
    cone_key = jax.random.PRNGKey(42)

    current_max_angle = np.radians(max_angle_deg)

    def rotation_matrix_from_vectors(vec1, vec2):
        """Create rotation matrix that rotates vec1 to vec2 (Rodrigues' formula)."""
        a = vec1 / (np.linalg.norm(vec1) + 1e-8)
        b = vec2 / (np.linalg.norm(vec2) + 1e-8)
        v = np.cross(a, b)
        c = np.dot(a, b)

        if np.linalg.norm(v) < 1e-8:
            if c > 0:
                return np.eye(3)
            else:
                # 180 degree rotation
                perp = np.array([1., 0., 0.]) if abs(a[0]) < 0.9 else np.array([0., 1., 0.])
                perp = perp - np.dot(perp, a) * a
                perp = perp / (np.linalg.norm(perp) + 1e-8)
                return 2 * np.outer(perp, perp) - np.eye(3)

        s = np.linalg.norm(v)
        kmat = np.array([[0, -v[2], v[1]],
                         [v[2], 0, -v[0]],
                         [-v[1], v[0], 0]])
        return np.eye(3) + kmat + kmat @ kmat * ((1 - c) / (s * s + 1e-8))

    for level in range(levels):
        n_theta = initial_div
        n_phi = initial_div * 2

        directions = []
        direction_results = []

        if level == 0:
            # Level 0: full sphere sampling
            for i in range(n_theta):
                theta_val = np.pi * (i / max(n_theta - 1, 1))
                for j in range(n_phi):
                    phi_val = 2 * np.pi * (j / n_phi)

                    direction = spherical_to_cartesian(theta_val, phi_val)
                    directions.append(direction)

                    track = ParticleParams(energy=energy_guess, position=optimal_position,
                                          theta=theta_val, phi=phi_val, t0=optimal_t0)
                    _, _, _, total_charge = prediction_simulator(track, cone_key)
                    loss = poisson_nll(observed_counts, total_charge)

                    direction_results.append({
                        'direction': np.array(direction),
                        'theta': float(theta_val),
                        'phi': float(phi_val),
                        'loss': float(loss)
                    })

                    if loss < best_loss:
                        best_loss = loss
                        best_direction = np.array(direction)
                        best_theta = theta_val
                        best_phi = phi_val
        else:
            # Level > 0: cone sampling around best direction
            # Create rotation matrix from z-axis to best direction
            z_axis = np.array([0., 0., 1.])
            R = rotation_matrix_from_vectors(z_axis, best_direction)

            for i in range(n_theta):
                # Cone angle from 0 to current_max_angle
                cone_theta = current_max_angle * (i / max(n_theta - 1, 1))
                for j in range(n_phi):
                    cone_phi = 2 * np.pi * (j / n_phi)

                    # Direction in local coords (cone around z-axis)
                    local_dir = np.array([
                        np.sin(cone_theta) * np.cos(cone_phi),
                        np.sin(cone_theta) * np.sin(cone_phi),
                        np.cos(cone_theta)
                    ])

                    # Rotate to global coords (cone around best_direction)
                    direction = R @ local_dir
                    direction = direction / (np.linalg.norm(direction) + 1e-8)

                    # Convert back to spherical for the track
                    theta_val, phi_val = cartesian_to_spherical(direction)

                    directions.append(direction)

                    track = ParticleParams(energy=energy_guess, position=optimal_position,
                                          theta=theta_val, phi=phi_val, t0=optimal_t0)
                    _, _, _, total_charge = prediction_simulator(track, cone_key)
                    loss = poisson_nll(observed_counts, total_charge)

                    direction_results.append({
                        'direction': np.array(direction),
                        'theta': float(theta_val),
                        'phi': float(phi_val),
                        'loss': float(loss)
                    })

                    if loss < best_loss:
                        best_loss = loss
                        best_direction = np.array(direction)
                        best_theta = theta_val
                        best_phi = phi_val

        search_path.append({
            'level': level,
            'directions': np.array(directions),
            'direction_results': direction_results,
            'best_direction': best_direction.copy(),
            'best_loss': best_loss
        })

        # Reduce cone angle for next level
        current_max_angle *= reduction

        if verbosity >= 2:
            print(f"  Level {level}: best_loss={best_loss:.6f}")

    # Calculate direction error
    cos_angle = np.clip(np.dot(best_direction, true_direction), -1.0, 1.0)
    direction_error = np.degrees(np.arccos(cos_angle))

    if verbosity >= 2:
        print(f"\n  Best direction: {best_direction}")
        print(f"  True direction: {true_direction}")
        print(f"  Direction error: {direction_error:.2f} degrees")

    return {
        'best_direction': best_direction,
        'best_theta': float(best_theta),
        'best_phi': float(best_phi),
        'best_loss': float(best_loss),
        'direction_error': direction_error,
        'search_path': search_path
    }

## Cell 9: Stage 3 - Energy Scan Optimization

In [ ]:
def run_stage_3_energy_scan_likelihood(optimal_position, best_theta, best_phi, optimal_t0,
                                       energy_guess, observed_times, observed_counts,
                                       true_energy, verbosity=2):
    """
    Stage 3: Energy scan at optimal position/direction.
    
    Uses energy_loss (log ratio of total counts) - only cares about matching
    total light yield, not spatial distribution.
    """
    if verbosity >= 2:
        print("\n" + "=" * 60)
        print("STAGE 3: Energy Scan Optimization (energy_loss)")
        print("=" * 60)
    
    energy_delta = config['energy_optimization']['energy_delta']
    n_steps = config['energy_optimization']['energy_scan_steps']
    
    energies = jnp.linspace(energy_guess - energy_delta, energy_guess + energy_delta, n_steps)
    
    scan_results = []
    best_loss = float('inf')
    best_energy = energy_guess
    initial_energy = energy_guess
    
    scan_key = jax.random.PRNGKey(42)
    
    for energy in energies:
        # Use energy_loss (log ratio) for energy scan
        track = ParticleParams(energy=energy, position=optimal_position,
                              theta=best_theta, phi=best_phi, t0=optimal_t0)
        _, _, _, total_charge = prediction_simulator(track, scan_key)
        loss = energy_loss(total_charge, observed_counts)
        
        scan_results.append({'energy': float(energy), 'loss': float(loss)})
        
        if loss < best_loss:
            best_loss = loss
            best_energy = energy
    
    energy_improvement = abs(best_energy - initial_energy)
    
    if verbosity >= 2:
        print(f"\n  Best energy: {best_energy:.1f} MeV (true: {true_energy:.1f} MeV)")
        print(f"  Energy error: {abs(best_energy - true_energy):.1f} MeV")
        print(f"  Improvement from initial: {energy_improvement:.1f} MeV")
    
    return {
        'best_energy': float(best_energy),
        'best_loss': float(best_loss),
        'energy_improvement': energy_improvement,
        'scan_results': scan_results
    }

print("Stage 3 functions defined (using energy_loss for energy scan).")

## Cell 10: Stage 4 - Adam Optimizer Refinement (Likelihood)

In [ ]:
def run_stage_4_adam_optimization_likelihood(initial_params, 
                                             observed_times, observed_counts,
                                             true_energy, true_position, true_direction, TRUE_T0,
                                             verbosity=2):
    """
    Stage 4: Adam optimizer refinement using likelihood-based loss.
    
    Uses combined loss with dynamic tau_vtx (poisson_nll + first_arrival_nll + vertex_loss)
    """
    if verbosity >= 2:
        print("\n" + "=" * 60)
        print("STAGE 4: Adam Optimizer Refinement (Likelihood + Vertex Loss)")
        print("=" * 60)
    
    # Convert true direction to spherical
    true_theta, true_phi = cartesian_to_spherical(true_direction)
    
    # Adam parameters from config
    ADAM_LEARNING_RATE = config['adam_optimizer']['learning_rate']
    ADAM_B1 = config['adam_optimizer']['b1']
    ADAM_B2 = config['adam_optimizer']['b2']
    ADAM_EPS = config['adam_optimizer']['eps']
    MAX_ITERATIONS = 400
    damping_factor = config['optimization_params']['damping_factor']
    tolerance = 1e-6
    
    # Learning-rate scaling
    POS_LR_SCALE = config['learning_rates']['position_learning_rate'] * 2.
    DIR_LR_SCALE = config['learning_rates']['direction_learning_rate'] * 5.
    T0_LR_SCALE = config['learning_rates']['t0_learning_rate']
    ENE_LR_SCALE = config['learning_rates']['energy_learning_rate']
    
    if verbosity >= 2:
        print(f"  Learning rate: {ADAM_LEARNING_RATE}")
        print(f"  Max iterations: {MAX_ITERATIONS}")
        print(f"  Initial params: {initial_params}")
    
    # Initialize optimizer
    optimizer = optax.adam(learning_rate=ADAM_LEARNING_RATE, b1=ADAM_B1, b2=ADAM_B2, eps=ADAM_EPS)
    opt_state = optimizer.init(initial_params)
    current_params = jnp.array(initial_params)
    
    # History tracking
    history = {
        'parameters': [current_params.copy()],
        'combined_losses': [],
        'charge_losses': [],
        'time_losses': [],
        'vertex_losses': [],
        'tau_vtx_values': [],
        'position_errors': [],
        'direction_errors': [],
        't0_errors': [],
        'energy_errors': [],
    }
    
    opt_key = jax.random.PRNGKey(12345)
    grad_norm = float('inf')
    
    adam_start_time = time.time()

    for iteration in range(MAX_ITERATIONS):
        opt_key, _ = jax.random.split(opt_key)
        
        # Compute loss and gradient using likelihood-based loss with vertex term
        (combined_loss, (charge_loss_val, time_loss_val, vtx_loss_val, sqrt_ct_val, tau_vtx_val)), grad = combined_grad_fn(
            current_params, observed_times, observed_counts, opt_key
        )
        
        # Handle NaN gradients
        if jnp.any(jnp.isnan(grad)):
            grad = jnp.nan_to_num(grad, nan=0.0)
        
        grad_norm = jnp.linalg.norm(grad)
        if grad_norm < tolerance:
            break

        # Phase 1: Direction only for first 25 iterations
        if iteration < 25:
            update_scales = jnp.array([0, 0, 0, 0, DIR_LR_SCALE, DIR_LR_SCALE, 0.])
        else:
            update_scales = jnp.array([
                POS_LR_SCALE, POS_LR_SCALE, POS_LR_SCALE,
                T0_LR_SCALE, DIR_LR_SCALE, DIR_LR_SCALE, ENE_LR_SCALE
            ])

        # Adam update with parameter-specific scaling
        updates, opt_state = optimizer.update(grad, opt_state, current_params)
        scaled_updates = updates * update_scales
        current_params = optax.apply_updates(current_params, scaled_updates)
        
        # Apply constraints
        current_params = jnp.array([
            jnp.clip(current_params[0], -DETECTOR_R * 0.95, DETECTOR_R * 0.95),
            jnp.clip(current_params[1], -DETECTOR_R * 0.95, DETECTOR_R * 0.95),
            jnp.clip(current_params[2], -DETECTOR_H/2 * 0.95, DETECTOR_H/2 * 0.95),
            jnp.clip(current_params[3], -20.0, 20.0),
            current_params[4],
            current_params[5],
            jnp.clip(current_params[6], 300.0, 2000.0)
        ])
        
        # Calculate errors
        current_position = current_params[:3]
        current_t0 = current_params[3]
        current_theta = current_params[4]
        current_phi = current_params[5]
        current_energy = current_params[6]
        current_direction = spherical_to_cartesian(current_theta, current_phi)
        
        position_error = float(jnp.linalg.norm(current_position - true_position))
        t0_error = float(abs(current_t0 - TRUE_T0))
        energy_error = float(abs(current_energy - true_energy))
        cos_angle = np.clip(np.dot(np.array(current_direction), np.array(true_direction)), -1.0, 1.0)
        direction_error = float(np.degrees(np.arccos(cos_angle)))
        
        # Store history
        history['parameters'].append(current_params.copy())
        history['combined_losses'].append(float(combined_loss))
        history['charge_losses'].append(float(charge_loss_val))
        history['time_losses'].append(float(time_loss_val))
        history['vertex_losses'].append(float(vtx_loss_val))
        history['tau_vtx_values'].append(float(tau_vtx_val))
        history['position_errors'].append(position_error)
        history['direction_errors'].append(direction_error)
        history['t0_errors'].append(t0_error)
        history['energy_errors'].append(energy_error)
        
        if verbosity >= 2 and ((iteration + 1) % 100 == 0 or iteration == 0):
            print(
                f"  Iter {iteration}: "
                f"loss={combined_loss:.6f} "
                f"(c={charge_loss_val:.4f}, t={time_loss_val:.4f}, v={vtx_loss_val:.4f}) "
                f"tau_vtx={tau_vtx_val:.3f} "
                f"pos_err={position_error:.3f}m, "
                f"dir_err={direction_error:.2f}deg, "
                f"t0_err={t0_error:.3f}, "
                f"E_err={energy_error:.1f}"
            )
    
    adam_end_time = time.time()
    adam_optimization_time = adam_end_time - adam_start_time
    
    # Final results
    final_position = current_params[:3]
    final_t0 = current_params[3]
    final_theta = current_params[4]
    final_phi = current_params[5]
    final_energy = current_params[6]
    final_direction = spherical_to_cartesian(final_theta, final_phi)
    
    final_position_error = float(jnp.linalg.norm(final_position - true_position))
    final_t0_error = float(abs(final_t0 - TRUE_T0))
    final_energy_error = float(abs(final_energy - true_energy))
    cos_angle = np.clip(np.dot(np.array(final_direction), np.array(true_direction)), -1.0, 1.0)
    final_direction_error = float(np.degrees(np.arccos(cos_angle)))
    
    if verbosity >= 2:
        print(f"\n  Optimization completed in {adam_optimization_time:.2f}s")
        print(f"  Final position error: {final_position_error:.3f} m")
        print(f"  Final direction error: {final_direction_error:.2f} deg")
        print(f"  Final t0 error: {final_t0_error:.3f}")
        print(f"  Final energy error: {final_energy_error:.1f} MeV")
        print(f"  Final tau_vtx: {history['tau_vtx_values'][-1]:.3f}")
    
    return {
        'initial_params': np.array(initial_params),
        'final_params': np.array(current_params),
        'final_position': np.array(final_position),
        'final_direction': np.array(final_direction),
        'final_theta': float(final_theta),
        'final_phi': float(final_phi),
        'final_t0': float(final_t0),
        'final_energy': float(final_energy),
        'final_position_error': final_position_error,
        'final_direction_error': final_direction_error,
        'final_t0_error': final_t0_error,
        'final_energy_error': final_energy_error,
        'final_combined_loss': history['combined_losses'][-1] if history['combined_losses'] else float('inf'),
        'final_charge_loss': history['charge_losses'][-1] if history['charge_losses'] else float('inf'),
        'final_time_loss': history['time_losses'][-1] if history['time_losses'] else float('inf'),
        'final_vertex_loss': history['vertex_losses'][-1] if history['vertex_losses'] else float('inf'),
        'final_tau_vtx': history['tau_vtx_values'][-1] if history['tau_vtx_values'] else 0.0,
        'adam_optimization_time': adam_optimization_time,
        'total_iterations': len(history['parameters']) - 1,
        'converged': grad_norm < tolerance,
        'history': history
    }


def visualize_stage_4(adam_results):
    """Visualization: Multi-panel loss and error curves"""
    history = adam_results['history']
    
    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    
    # Combined loss
    axes[0, 0].semilogy(history['combined_losses'])
    axes[0, 0].set_title('Combined Loss')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Position error
    axes[0, 1].plot(history['position_errors'])
    axes[0, 1].set_title('Position Error (m)')
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Direction error
    axes[0, 2].plot(history['direction_errors'])
    axes[0, 2].set_title('Direction Error (deg)')
    axes[0, 2].set_xlabel('Iteration')
    axes[0, 2].grid(True, alpha=0.3)
    
    # tau_vtx values
    axes[0, 3].plot(history['tau_vtx_values'])
    axes[0, 3].set_title('tau_vtx (dynamic)')
    axes[0, 3].set_xlabel('Iteration')
    axes[0, 3].grid(True, alpha=0.3)
    
    # t0 error
    axes[1, 0].plot(history['t0_errors'])
    axes[1, 0].set_title('t0 Error')
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Energy error
    axes[1, 1].plot(history['energy_errors'])
    axes[1, 1].set_title('Energy Error (MeV)')
    axes[1, 1].set_xlabel('Iteration')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Component losses
    axes[1, 2].semilogy(history['charge_losses'], label='Charge (Poisson NLL)')
    axes[1, 2].semilogy(history['time_losses'], label='Time (First-arrival NLL)')
    axes[1, 2].semilogy(history['vertex_losses'], label='Vertex Loss')
    axes[1, 2].legend()
    axes[1, 2].set_title('Component Losses')
    axes[1, 2].set_xlabel('Iteration')
    axes[1, 2].grid(True, alpha=0.3)
    
    # sqrt(c*t) metric
    sqrt_ct = [np.sqrt(c*t) for c, t in zip(history['charge_losses'], history['time_losses'])]
    axes[1, 3].semilogy(sqrt_ct)
    axes[1, 3].set_title('sqrt(charge * time)')
    axes[1, 3].set_xlabel('Iteration')
    axes[1, 3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    return fig

print("Stage 4 functions defined (with vertex loss and dynamic tau_vtx).")

## Cell 11: Main Processing Loop

In [ ]:
# =====================================================================
# Main Processing Loop
# =====================================================================

# Storage for results
all_event_results = []

# Performance tracking
energy_guess_errors = []
grid_position_errors = []
grid_t0_errors = []
cone_direction_errors = []
energy_scan_improvements = []
final_position_errors = []
final_direction_errors = []
final_t0_errors = []
final_energy_errors = []
final_combined_losses = []
final_charge_losses = []
final_time_losses = []
convergence_rates = []

# Generate random keys
main_key = jax.random.PRNGKey(42)
event_keys = jax.random.split(main_key, N_EVENTS)

print(f"Processing {N_EVENTS} events with likelihood-based loss...")
print("=" * 80)

for event_idx in range(N_EVENTS):
    event_start_time = time.time()
    
    print(f"\n{'='*80}")
    print(f"EVENT {event_idx}")
    print(f"{'='*80}")
    
    try:
        # Generate event with endpoint validation
        event_key = event_keys[event_idx]
        max_attempts = 10
        endpoint_valid = False
        
        for attempt in range(max_attempts):
            event_data = generate_event_data(
                event_idx, event_key, data_dir,
                data_simulator, detector_bounds, fraction=0.9
            )
            
            endpoint_valid = check_track_endpoint_in_detector(
                event_data['true_position'],
                event_data['true_direction'],
                event_data['true_energy'],
                range_params, detector_bounds, fraction=0.9
            )
            
            if endpoint_valid:
                break
            event_key, _ = jax.random.split(event_key)
        
        if not endpoint_valid:
            print(f"  ERROR: Could not generate valid event after {max_attempts} attempts")
            continue
        
        # Extract event data
        true_position = event_data['true_position']
        true_direction = event_data['true_direction']
        true_energy = event_data['true_energy']
        TRUE_T0 = event_data['TRUE_T0']
        true_data = event_data['true_data']
        
        # Get hit information
        hit_mask = event_data['hit_counts'] > -999
        hit_detector_positions = detector_points[hit_mask]
        observed_times = event_data['hit_times'][hit_mask]
        observed_counts = event_data['hit_counts'][hit_mask]
        
        # For likelihood loss, we need all sensors (not just hit ones)
        all_observed_times = event_data['hit_times']
        all_observed_counts = event_data['hit_counts']
        
        print(f"  True position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}]")
        print(f"  True direction: [{true_direction[0]:.3f}, {true_direction[1]:.3f}, {true_direction[2]:.3f}]")
        print(f"  True energy: {true_energy:.1f} MeV")
        print(f"  True t0: {TRUE_T0:.3f}")
        print(f"  N hits: {int(jnp.sum(hit_mask))}")
        
        # ===== STAGE 0: Energy estimation =====
        stage0_results = run_stage_0_energy_estimation_likelihood(
            all_observed_times, all_observed_counts,
            true_energy, verbosity=1
        )
        
        # ===== STAGE 1: Position + t0 grid search =====
        stage1_results = run_stage_1_position_search(
            hit_detector_positions, observed_times, observed_counts,
            true_position, TRUE_T0, initial_t0=0.0, verbosity=1
        )
        
        if stage1_results['best_position'] is None:
            print(f"  ERROR: Stage 1 failed")
            continue
        
        # ===== STAGE 2: Direction cone search =====
        stage2_results = run_stage_2_direction_search_likelihood(
            stage1_results['best_position'], stage1_results['best_t0'],
            stage0_results['best_energy'], 
            all_observed_times, all_observed_counts, 
            true_direction, verbosity=1
        )
        
        # ===== STAGE 3: Energy scan =====
        stage3_results = run_stage_3_energy_scan_likelihood(
            stage1_results['best_position'], stage2_results['best_theta'],
            stage2_results['best_phi'], stage1_results['best_t0'],
            stage0_results['best_energy'], 
            all_observed_times, all_observed_counts,
            true_energy, verbosity=1
        )

        # ===== STAGE 4: Adam optimization =====
        true_theta, true_phi = cartesian_to_spherical(true_direction)

        # initial_params = jnp.array([
        #     stage1_results['best_position'][0],
        #     stage1_results['best_position'][1],
        #     stage1_results['best_position'][2],
        #     stage1_results['best_t0'],
        #     true_theta,
        #     true_phi,
        #     true_energy,#stage3_results['best_energy']
        # ])

        initial_params = jnp.array([
            stage1_results['best_position'][0],
            stage1_results['best_position'][1],
            stage1_results['best_position'][2],
            stage1_results['best_t0'],
            stage2_results['best_theta'],
            stage2_results['best_phi'],
            stage3_results['best_energy']
        ])

        # initial_params = jnp.array([
        #     true_position[0],
        #     true_position[1],
        #     true_position[2],
        #     TRUE_T0,#stage1_results['best_t0'],
        #     true_theta,
        #     true_phi,
        #     true_energy,#stage3_results['best_energy']
        # ])
        
        # # ===== STAGE 4: Adam optimization =====
        # initial_params = jnp.array([
        #     stage1_results['best_position'][0],
        #     stage1_results['best_position'][1],
        #     stage1_results['best_position'][2],
        #     TRUE_T0,#stage1_results['best_t0'],
        #     stage2_results['best_theta'],
        #     stage2_results['best_phi'],
        #     stage3_results['best_energy']
        # ])
        
        stage4_results = run_stage_4_adam_optimization_likelihood(
            initial_params, 
            all_observed_times, all_observed_counts,
            true_energy, true_position, true_direction, TRUE_T0,
            verbosity=2
        )
        
        # Store results
        event_end_time = time.time()
        
        event_result = {
            'event_data': {
                'event_idx': event_data['event_idx'],
                'true_energy': event_data['true_energy'],
                'true_position': event_data['true_position'],
                'true_direction': event_data['true_direction'],
                'TRUE_T0': event_data['TRUE_T0'],
                'true_data': event_data['true_data'],
                'hit_detector_positions': hit_detector_positions,
                'observed_times': observed_times,
                'observed_counts': observed_counts,
            },
            'stage0': stage0_results,
            'stage1': stage1_results,
            'stage2': stage2_results,
            'stage3': stage3_results,
            'stage4': stage4_results,
            'optimization_results': stage4_results,
            'total_event_time': event_end_time - event_start_time
        }
        all_event_results.append(event_result)
        
        # Track metrics
        energy_guess_errors.append(abs(stage0_results['best_energy'] - true_energy))
        grid_position_errors.append(stage1_results['position_error'])
        grid_t0_errors.append(stage1_results['t0_error'])
        cone_direction_errors.append(stage2_results['direction_error'])
        energy_scan_improvements.append(stage3_results['energy_improvement'])
        final_position_errors.append(stage4_results['final_position_error'])
        final_direction_errors.append(stage4_results['final_direction_error'])
        final_t0_errors.append(stage4_results['final_t0_error'])
        final_energy_errors.append(stage4_results['final_energy_error'])
        final_combined_losses.append(stage4_results['final_combined_loss'])
        final_charge_losses.append(stage4_results['final_charge_loss'])
        final_time_losses.append(stage4_results['final_time_loss'])
        convergence_rates.append(1.0 if stage4_results['converged'] else 0.0)
        
        print(f"\n  Event {event_idx} completed in {event_end_time - event_start_time:.2f}s")
        print(f"  Final: pos_err={stage4_results['final_position_error']:.3f}m, "
              f"dir_err={stage4_results['final_direction_error']:.2f}deg, "
              f"t0_err={stage4_results['final_t0_error']:.3f}, "
              f"E_err={stage4_results['final_energy_error']:.1f}MeV")
        
    except Exception as e:
        print(f"  ERROR processing event {event_idx}: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*80}")
print(f"Completed processing {len(all_event_results)} events successfully")
print(f"{'='*80}")

## Cell 12: Performance Summary

In [ ]:
#### Performance summary
if len(all_event_results) > 0:
    print("\n")
    performance_summary(
        energy_guess_errors,
        grid_position_errors,
        cone_direction_errors,
        energy_scan_improvements,
        final_position_errors,
        final_direction_errors,
        final_t0_errors,
        final_energy_errors,
        final_combined_losses,
        final_charge_losses,  # Using charge_losses instead of vertex_losses
        final_time_losses,    # Using time_losses instead of counts_losses
        [],  # final_energy_losses not tracked separately
        convergence_rates,
    )

    # t0 grid search statistics
    print("\n" + "=" * 80)
    print("4D Grid Search t0 Performance:")
    print("=" * 80)
    grid_t0_errors_arr = np.array(grid_t0_errors)
    print(f"Mean t0 error:   {np.mean(grid_t0_errors_arr):.4f}")
    print(f"Median t0 error: {np.median(grid_t0_errors_arr):.4f}")
    print(f"Std t0 error:    {np.std(grid_t0_errors_arr):.4f}")

    # Timing summary
    print("\n" + "=" * 80)
    print("TIMING SUMMARY")
    print("=" * 80)
    total_times = [e['total_event_time'] for e in all_event_results]
    adam_times = [e['stage4']['adam_optimization_time'] for e in all_event_results]
    print(f"Total event time - Mean: {np.mean(total_times):.2f}s, Median: {np.median(total_times):.2f}s")
    print(f"Adam time - Mean: {np.mean(adam_times):.2f}s, Median: {np.median(adam_times):.2f}s")
    print(f"Adam % of total: {100 * np.sum(adam_times) / np.sum(total_times):.1f}%")
else:
    print("No events processed successfully.")

## Cell 13: Visualize Single Event

In [ ]:
# Select event to visualize
EVENT_TO_VISUALIZE = 0

if len(all_event_results) > EVENT_TO_VISUALIZE:
    event = all_event_results[EVENT_TO_VISUALIZE]
    true_energy = event['event_data']['true_energy']
    
    print(f"\nVisualizing all stages for Event {EVENT_TO_VISUALIZE}")
    print("=" * 60)
    
    # Stage 0
    print("\nStage 0: Energy estimation (Likelihood)")
    visualize_stage_0(event['stage0'], true_energy)
    
    # Stage 4
    print("\nStage 4: Adam optimization (Likelihood)")
    visualize_stage_4(event['stage4'])
else:
    print(f"Event {EVENT_TO_VISUALIZE} not available.")

## Cell 14: Save Results

In [ ]:
# Save results to pickle
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / f'tracking_opt_likelihood_{N_EVENTS}_events.pkl'

results_to_save = {
    'all_event_results': all_event_results,
    'config': config,
    'metrics': {
        'energy_guess_errors': energy_guess_errors,
        'grid_position_errors': grid_position_errors,
        'grid_t0_errors': grid_t0_errors,
        'cone_direction_errors': cone_direction_errors,
        'energy_scan_improvements': energy_scan_improvements,
        'final_position_errors': final_position_errors,
        'final_direction_errors': final_direction_errors,
        'final_t0_errors': final_t0_errors,
        'final_energy_errors': final_energy_errors,
        'final_combined_losses': final_combined_losses,
        'final_charge_losses': final_charge_losses,
        'final_time_losses': final_time_losses,
        'convergence_rates': convergence_rates
    },
    'loss_type': 'likelihood',
    'tau': TAU
}

with open(output_file, 'wb') as f:
    pickle.dump(results_to_save, f)

print(f"Results saved to: {output_file}")